# Análisis de Información Mutua en Alta Dimensión

Este cuaderno presenta técnicas computacionales para analizar
dependencias informacionales en sistemas multivariados de alta dimensión.

Objetivos:

- Simular sistemas con dependencias estructuradas
- Estimar información mutua entre múltiples variables
- Construir matrices de información mutua
- Analizar información mutua condicional
- Evaluar desafíos computacionales en alta dimensión


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler


## Generación de Datos Sintéticos Multivariados

Construiremos un sistema con:

- dependencias directas
- relaciones no lineales
- correlaciones indirectas


In [ ]:
np.random.seed(42)

n_samples = 1500

X1 = np.random.normal(size=n_samples)
X2 = 0.8 * X1 + np.random.normal(scale=0.4, size=n_samples)
X3 = np.sin(X1) + np.random.normal(scale=0.2, size=n_samples)
X4 = X2**2 + np.random.normal(scale=0.3, size=n_samples)
X5 = np.random.normal(size=n_samples)

data = np.vstack([X1, X2, X3, X4, X5]).T

df = pd.DataFrame(data, columns=["X1","X2","X3","X4","X5"])

df.head()


## Normalización

La normalización estabiliza estimaciones informacionales
en espacios de alta dimensión.


In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df.values)


## Estimación de Matriz de Información Mutua


In [ ]:
n_features = data_scaled.shape[1]
mi_matrix = np.zeros((n_features, n_features))

for i in range(n_features):
    for j in range(n_features):
        if i != j:
            mi_matrix[i, j] = mutual_info_regression(
                data_scaled[:, [i]],
                data_scaled[:, j],
                random_state=42
            )[0]

mi_df = pd.DataFrame(mi_matrix,
                     columns=df.columns,
                     index=df.columns)

mi_df


## Visualización de la Matriz MI


In [ ]:
plt.figure()

plt.imshow(mi_matrix)
plt.colorbar()

plt.xticks(range(n_features), df.columns)
plt.yticks(range(n_features), df.columns)

plt.title("Matriz de Información Mutua")
plt.show()


## Información Mutua Condicional Aproximada

Aproximaremos:

I(Xi ; Xj | Xk)

mediante regresión residual.


In [ ]:
from sklearn.linear_model import LinearRegression

def conditional_mi_residual(X, Y, Z):
    model_x = LinearRegression().fit(Z, X)
    model_y = LinearRegression().fit(Z, Y)

    rx = X - model_x.predict(Z)
    ry = Y - model_y.predict(Z)

    return mutual_info_regression(
        rx.reshape(-1,1),
        ry,
        random_state=42
    )[0]


In [ ]:
cmiv = conditional_mi_residual(
    data_scaled[:,0],
    data_scaled[:,3],
    data_scaled[:,1].reshape(-1,1)
)

print("I(X1 ; X4 | X2) ≈", cmiv)


## Crecimiento de Dimensionalidad

Analizamos el comportamiento de la estimación
cuando aumenta el número de variables.


In [ ]:
dims = []
times = []

for d in [5,10,20,30]:
    X = np.random.normal(size=(800,d))

    import time
    t0 = time.time()

    for i in range(d):
        for j in range(d):
            if i != j:
                mutual_info_regression(X[:,[i]], X[:,j])

    t1 = time.time()

    dims.append(d)
    times.append(t1 - t0)


In [ ]:
plt.figure()
plt.plot(dims, times, marker="o")

plt.xlabel("Dimensión")
plt.ylabel("Tiempo (s)")
plt.title("Costo Computacional en Alta Dimensión")

plt.show()


## Discusión

Observaciones principales:

- La información mutua detecta dependencias no lineales.
- La dimensionalidad incrementa rápidamente el costo computacional.
- La información mutua condicional permite eliminar dependencias indirectas.
- La estimación robusta en alta dimensión requiere métodos avanzados.

Este análisis establece las bases para aplicaciones reales
en ciencia de datos y aprendizaje profundo.

{cite}`kraskov2004mi`.